# Video Preprocessing

This file is meant to be an initial example on how we can preprocess video data, if the task requires analyzing swing from video.

This file can also be found as a python source code file, running either would be fine.

In [9]:
import os
import cv2
import csv
import json
import pandas as pd
from pathlib import Path
import mediapipe as mp
#import swingtrace as st
from ultralytics import YOLO

### Finding Video Recordings

In [2]:
def find_recordings(root):
    recordings = []
    for video_path in sorted(root.rglob("swing*.mp4")):
        print(f"Found video: {video_path}")
        recordings.append(video_path)
    return recordings

In [3]:
root = Path(r"own_data")
records = find_recordings(root)
print(f"{len(records)} videos were found in the root path: '{root}'")

Found video: own_data\swing1.mp4
Found video: own_data\swing2.mp4
Found video: own_data\swing3.mp4
3 videos were found in the root path: 'own_data'


### Utilize MediaPipe for Landmark detection

Google MediaPipe has a landmark detection model. This has now been done with the lite model, but the heavier models can also be downloaded and used instead.

The result from running this (or .py) file is a CSV file containing landmarks from all videos. For this smaller example data I thought it would be neat to see them on the video, so annotated_swing*.mp4 contains the landmarks on top of the original videos.

In [4]:
def open_video(video_path):
    # Open the video file
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    #frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("End of video or failed to grab frame.")
            break  # End of video
            
        # OpenCV reads BGR, but MediaPipe needs RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Wrap into MP image object
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # Get timestamp in ms
        timestamp_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))

        yield frame, mp_image, timestamp_ms
    
    cap.release()

In [15]:
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="pose_landmarker_lite.task"),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1,
)

club_model = YOLO("swingtrace-yolo11m.pt")

with open("all_pose_landmarks.csv", mode="w", newline="") as csv_file:
    writer = csv.writer(csv_file)
    
    # Write CSV header
    header = ["video_name", "frame_idx", "timestamp_ms"]
    for i in range(33):
        header.extend([f"landmark_{i}_x", f"landmark_{i}_y", f"landmark_{i}_z", f"landmark_{i}_vis"])
    header.extend(["club_head_x", "club_head_y", "club_head_detected"])
    writer.writerow(header)


    for video in records:
        print(f"Processing file: {video}")
        video_path = Path(video)

        # SwingTrace
        print("Running Club Tracking...")
        club_data_by_frame = {}
        result = club_model.track(source=video_path, conf=0.25, device="mps", stream=True, persist=True)
        #result = st.track(video_path, mode="both", device="mps")  # "cpu", "cuda" or "mps"
        #print(result.summary())
        #club_data_by_frame = {frame.idx: frame.club_coords for frame in result.frames}

        for frame_idx, res in enumerate(result):
            detected = False
            cx, cy = None, None
            if res.boxes is not None and len(res.boxes) > 0:
                # Extract the highest confidence box (or filter by class if desired)
                boxes = res.boxes.xyxy.cpu().numpy()
                classes = res.boxes.cls.cpu().numpy()
                confs = res.boxes.conf.cpu().numpy()
                # Filter for Club Head specifically
                head_indices = [i for i, cls_id in enumerate(classes) if int(cls_id) == 1]
                if len(head_indices) > 0:
                    best_idx = head_indices[confs[head_indices].argmax()]
                    box = boxes[best_idx]
                    x1, y1, x2, y2 = map(int, box[:4])
                    
                    # Center of the bounding box
                    cx = float((x1 + x2) / 2.0)
                    cy = float((y1 + y2) / 2.0)
                    detected = True
                    club_data_by_frame[frame_idx] = {
                        "x": cx,
                        "y": cy,
                        "detected": detected,
                        "box": (x1, y1, x2, y2)
                    }
       
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
        output_video_path = f"annotated_{video_path.name}"
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
        
        with PoseLandmarker.create_from_options(options) as landmarker:
            frame_idx = 0 
            for frame, mp_image, timestamp_ms in open_video(video):
                result = landmarker.detect_for_video(mp_image, timestamp_ms)

                row = [video_path.name, frame_idx, timestamp_ms]
                
                if result.pose_landmarks:
                    landmarks = result.pose_landmarks[0] # First person detected
                    
                    # Write row to CSV
                    for lm in landmarks:
                        row.extend([lm.x, lm.y, lm.z, lm.visibility])
                    
    
                    # Draw landmarks on video
                    for lm in landmarks:
                        cx, cy = int(lm.x * width), int(lm.y * height)
                        cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)

                # Club data lookup for current frame idx
                club_info = club_data_by_frame.get(frame_idx, {"x": None, "y": None, "detected": False})
                
                # Append Club info to CSV row
                row.extend([club_info["x"], club_info["y"], club_info["detected"]])

                writer.writerow(row)

                # Draw Club Tracking Visuals on frame
                if club_info["detected"]:
                    x1, y1, x2, y2 = club_info["box"]
                    # Draw box around the club head (Cyan/Blue box)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    # Draw center dot on the club head
                    cv2.circle(frame, (int(club_info["x"]), int(club_info["y"])), 5, (0, 0, 255), -1)
                    cv2.putText(frame, "Club Head", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
                
                out.write(frame)
                frame_idx += 1
        cap.release()
        out.release()
        print(f"Finished & saved: {output_video_path}\n")
print("All videos processed successfully!")     

Processing file: own_data\swing1.mp4
Running Club Tracking...

video 1/1 (frame 1/128) C:\Users\katar\OneDrive\Documents\AI-Hackathon-2026\porokoodarit-sinceai\own_data\swing1.mp4: 640x384 1 0, 1 1, 1 3, 555.0ms
video 1/1 (frame 2/128) C:\Users\katar\OneDrive\Documents\AI-Hackathon-2026\porokoodarit-sinceai\own_data\swing1.mp4: 640x384 1 0, 1 1, 1 3, 636.6ms
video 1/1 (frame 3/128) C:\Users\katar\OneDrive\Documents\AI-Hackathon-2026\porokoodarit-sinceai\own_data\swing1.mp4: 640x384 1 0, 1 1, 1 3, 652.5ms
video 1/1 (frame 4/128) C:\Users\katar\OneDrive\Documents\AI-Hackathon-2026\porokoodarit-sinceai\own_data\swing1.mp4: 640x384 1 0, 1 1, 1 3, 642.0ms
video 1/1 (frame 5/128) C:\Users\katar\OneDrive\Documents\AI-Hackathon-2026\porokoodarit-sinceai\own_data\swing1.mp4: 640x384 1 0, 1 1, 1 3, 674.1ms
video 1/1 (frame 6/128) C:\Users\katar\OneDrive\Documents\AI-Hackathon-2026\porokoodarit-sinceai\own_data\swing1.mp4: 640x384 1 0, 1 1, 1 3, 678.2ms
video 1/1 (frame 7/128) C:\Users\katar\OneD